# Avance 3 — Baseline (Equipo 17, AgroSatCopilot)

## Proyecto Integrador MNA · Tec de Monterrey

**Equipo 17**

- Carlos Isaac Ávila Gutiérrez — A01796035
- Carlos Aaron Bocanegra Buitrón — A01796345
- Arthur Jafed Zizumbo Velasco — A01796363

**Curso**: MNA — Tec de Monterrey · 20-abr → 3-jul-2026

**Sponsor académico**: Dr. Gerardo José Camacho — gjcamacho@tec.mx

**Fecha de entrega**: 2026-05-20 (con correcciones post-A3 cerradas 2026-05-27).

---

## Resumen ejecutivo

Este cuaderno **concentra y consolida** el trabajo del Avance 3 leyendo los artefactos generados por las cinco libretas previas. No reentrena modelos: lee tablas, lee figuras, ejecuta una sola llamada a `select_winning_features` para nombrar el conjunto ganador y entrega las respuestas a las **cinco preguntas oficiales del Avance 3** con cifras consolidadas.

Las cinco preguntas oficiales son:

1. **¿Qué algoritmo se puede utilizar como baseline** para predecir las variables objetivo?
2. **¿Se puede determinar la importancia de las características** para el modelo generado? (incluir características irrelevantes afecta el rendimiento y aumenta la complejidad).
3. **¿El modelo está sub/sobreajustando** los datos de entrenamiento?
4. **¿Cuál es la métrica adecuada** para este problema de negocio?
5. **¿Cuál debería ser el desempeño mínimo** a obtener?

## Estructura

1. Las 5 preguntas oficiales y dónde se contesta cada una.
2. P1 — Tabla consolidada de los 5 modelos (3 tabulares + 2 temporales).
3. P2 — Importancia de características y bloques irrelevantes.
4. P3 — Diagnóstico sub/sobreajuste para los 5 modelos.
5. P4 — Justificación de F1-macro como métrica principal.
6. P5 — Desempeño mínimo: target 0.60, estado actual y plan.
7. Comparativa AlphaEarth vs Sentinel-2 crudo vs vector combinado.
8. Resultados FarSLIP vs RemoteCLIP.
9. Clustering fenológico sin coordenadas.
10. Tabla H-1..H-4 con decisiones consolidadas (cuatro hipótesis del Avance 3).
11. Conjunto ganador: única llamada a `select_winning_features` + manifest JSON.
12. Trazabilidad MLflow consolidada (todos los `run_id`).
13. Referencias.


In [ ]:
COMPARISON_PATH_04 = "reports/baseline/04_baseline/model_comparison_04.parquet"
COMPARISON_PATH_04B = "reports/baseline/04b_baseline/model_comparison_04b.parquet"
ABLATION_BASE_PATH_04C = "reports/baseline/04c_baseline/ablation_table.parquet"
ABLATION_OPTIONAL_PATH_05 = "reports/baseline/05_reencuadre/ablation_table.parquet"
COMPARISON_TEMPORAL_PATH = "reports/baseline/05_reencuadre/model_comparison_temporal.parquet"
COMPARISON_SCENARIOS_PATH = "reports/baseline/04_baseline/comparison_alphaearth_vs_s2.csv"
FUSED_PATH = "data/features/features_fused_italy.parquet"
WINNING_OUTPUT = "data/features/features_fused_winning_italy.parquet"
FIGURES_SUBDIR = "us-023-preview/Avance3"
REPORTS_SUBDIR = "baseline/Avance3"
PROMOTE_THRESHOLD = 0.005
F1_TARGET = 0.60


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))

# Chdir al repo root para que las rutas relativas de la celda `parameters`
# (FEATURES_PATH = "data/...", etc.) resuelvan igual sin importar desde donde
# se haya lanzado el kernel (VS Code abre con cwd = carpeta del notebook).
# Esto preserva el contrato papermill (parametros como strings relativas) y
# elimina FileNotFoundError causado por cwd != repo root.
os.chdir(env.repo)
display(Markdown(f"**cwd anclado al repo root**: `{env.repo}`"))


## 1. Las cinco preguntas oficiales y dónde se contestan

| # | Pregunta oficial | Notebook(s) que aportan la respuesta |
|---|------------------|--------------------------------------|
| **P1** | ¿Qué algoritmo se puede utilizar como baseline? | `04_baseline` (3 tabulares + escenarios) · `04b_baseline` (piloto + cifras US-022b) · `04c_baseline` (qué bloques aportan) · `05_reencuadre` (5 modelos: 3 tabulares + 2 temporales sobre conjunto ganador) |
| **P2** | ¿Se puede determinar la importancia de las características? | `04_baseline` §5 (Gini, gain, SHAP, dominancia AE) · `04c_baseline` (ablación: bloques irrelevantes) · `05_reencuadre` §3 (bloques opcionales) · `04_farslip_eval` (separabilidad por espacio de embedding) |
| **P3** | ¿El modelo sub/sobreajusta? | `04_baseline` §10 (learning curves + `diagnose_fit` para RF/XGB/LGBM) · `05_reencuadre` §5.1 (loss train vs val por época + `diagnose_temporal_fit` para TempCNN/InceptionTime) |
| **P4** | ¿Cuál es la métrica adecuada? | `04_baseline` §2 (justificación F1-macro con desbalance 31×) |
| **P5** | ¿Desempeño mínimo a obtener? | `04_baseline` §3 (justificación 0.60: random vs trivial vs U-TAE PASTIS) |

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from ml.eval.reencuadre_plots import plot_optional_blocks_ablation
from ml.eval.feature_ablation import FeatureAblationResult

# Helper compacto para leer parquet con fallback explicito.
def _read_or_skip(path_str: str, label: str) -> pl.DataFrame | None:
    p = Path(path_str)
    if not p.exists():
        display(Markdown(
            f'> `{label}` no encontrado en `{path_str}`. '
            'Ejecuta el notebook correspondiente antes de re-ejecutar Avance3.'
        ))
        return None
    return pl.read_parquet(p)


## 2. P1 — Algoritmo baseline: tabla consolidada de los 5 modelos

Leemos la tabla del `04_baseline.ipynb` (3 tabulares: RF, XGB, LGBM) y la del `05_reencuadre_fenologico.ipynb` (5 modelos sobre el conjunto ganador). El ranking final por F1-macro decide qué modelo promovemos como **baseline canónico** del Avance 3 y cuáles quedan como aprendices base para los ensambles del Avance 5.

In [ ]:
comparison_04 = _read_or_skip(COMPARISON_PATH_04, 'model_comparison_04')
comparison_temporal = _read_or_skip(
    COMPARISON_TEMPORAL_PATH, 'model_comparison_temporal (5 modelos)'
)

if comparison_04 is not None:
    display(Markdown('**Tabla `04_baseline` — 3 modelos tabulares sobre conjunto fused completo**:'))
    display(comparison_04.sort('f1_macro', descending=True))
if comparison_temporal is not None:
    display(Markdown('**Tabla `05_reencuadre` — 5 modelos sobre conjunto ganador post-ablación**:'))
    display(comparison_temporal)
    best_row = comparison_temporal.row(0, named=True)
    display(Markdown(
        f'**Modelo ganador del Avance 3**: `{best_row["model"]}` '
        f'(familia: `{best_row["family"]}`) con F1-macro = `{best_row["f1_macro"]:.4f}`.'
    ))


## 3. P2 — Importancia de las características y bloques irrelevantes

Dos vistas complementarias:

- **Por feature individual** (gini, gain, SHAP, dominancia AlphaEarth): ver `04_baseline.ipynb` §5 + figuras `shap_summary_*.png`, `feature_importance_*.png` en `paper/figures/us-023-preview/04_baseline/`.
- **Por bloque completo** (ablación): leemos las tablas de `04c_baseline` (bloques base) y `05_reencuadre` (bloques opcionales) y graficamos el aporte agregado.

In [ ]:
ablation_base = _read_or_skip(ABLATION_BASE_PATH_04C, 'ablation_table_base (04c)')
ablation_optional = _read_or_skip(ABLATION_OPTIONAL_PATH_05, 'ablation_table (05)')

if ablation_base is not None:
    display(Markdown('**Ablación base (`04c_baseline`)** — qué bloques del fused aportan:'))
    display(ablation_base.sort('f1_macro', descending=True))

if ablation_optional is not None:
    display(Markdown('**Ablación completa (`05_reencuadre`)** — con bloques opcionales:'))
    display(ablation_optional.sort('f1_macro', descending=True))
    results = [
        FeatureAblationResult(
            feature_set=row['feature_set'],
            model_kind=row['model'],
            f1_macro=row['f1_macro'] if row['f1_macro'] is not None else float('nan'),
            f1_weighted=row['f1_weighted'] if row['f1_weighted'] is not None else float('nan'),
            miou=row['miou'] if row['miou'] is not None else float('nan'),
            n_features=row['n_features'],
            delta_vs_full=row['delta_vs_full'] if row['delta_vs_full'] is not None else float('nan'),
        )
        for row in ablation_optional.iter_rows(named=True)
    ]
    fig = plot_optional_blocks_ablation(results)
    fig.savefig(env.figures_dir / 'optional_blocks.png', bbox_inches='tight')
    display(fig)
    plt.close(fig)


## 4. P3 — Sub/sobreajuste para los 5 modelos

Los 3 modelos tabulares (RF, XGB, LGBM) se diagnostican en `04_baseline.ipynb` §10 con `diagnose_fit` sobre la curva de aprendizaje clásica. Los 2 modelos temporales (TempCNN, InceptionTime) se diagnostican en `05_reencuadre.ipynb` §5.1 con `diagnose_temporal_fit` sobre el historial de train_loss/val_loss leído desde MLflow.

Aquí incrustamos las imágenes generadas por ambos cuadernos.

In [ ]:
from IPython.display import Image

lc_figures = [
    ('Curva aprendizaje RF', 'paper/figures/us-023-preview/04_baseline/learning_curve_rf.png'),
    ('Curva aprendizaje XGB', 'paper/figures/us-023-preview/04_baseline/learning_curve_xgb.png'),
    ('Curva loss TempCNN', 'paper/figures/us-023-preview/05_reencuadre/loss_history_tempcnn.png'),
    ('Curva loss InceptionTime', 'paper/figures/us-023-preview/05_reencuadre/loss_history_inceptiontime.png'),
]
for label, rel_path in lc_figures:
    full_path = env.repo / rel_path
    if full_path.exists():
        display(Markdown(f'**{label}** — `{rel_path}`:'))
        display(Image(filename=str(full_path)))
    else:
        display(Markdown(f'> `{label}` no disponible (`{rel_path}` no existe).'))


## 5. P4 — Métrica adecuada para este problema

**F1-macro** es la métrica principal del Avance 3. Justificación en lenguaje accesible:

- **Desbalance fuerte** (~31× max/min): la accuracy puede dar lecturas engañosamente altas si el modelo predice solo las clases mayoritarias. Un clasificador trivial que asigna siempre la clase mayoritaria daría accuracy ≈ 25-30% pero F1-macro cercano a cero — cualquier modelo razonable tiene que superar eso con claridad.
- **Importan todas las clases por igual** (cada cultivo cuenta para el campesino que lo siembra): F1-macro promedia el F1 por clase sin ponderar por soporte, así que penaliza los fallos en clases minoritarias. F1-weighted, en cambio, las disimula.
- **Kappa** mide acuerdo corregido por azar pero no separa el comportamiento por clase — útil como medida global, insuficiente como única.
- **mIoU (Jaccard macro)** es equivalente a F1-macro en sensibilidad al desbalance, más estricto. Se reporta como segunda métrica.

**Decisión documentada**: principal F1-macro, secundaria mIoU; accuracy y kappa se reportan como contexto.

## 6. P5 — Desempeño mínimo a obtener

**Target acordado con el sponsor**: F1-macro ≥ 0.60 sobre las 18 clases PASTIS con validación cruzada espacial. Justificación con tres referencias:

- **Punto de partida (random)**: clasificador aleatorio uniforme sobre 18 clases ≈ F1-macro 0.06; predicción trivial de la clase mayoritaria ≈ 0.03.
- **Estado del arte**: PASTIS-R con **U-TAE** (Garnot et al. 2021) reporta mIoU ~0.65; con DINOv3 + linear probe se alcanzan ~0.55-0.60.
- **Decisión del equipo**: F1-macro ≥ 0.60 sobre 18 clases con CV espacial es el mínimo publicable. Si no se alcanza con el baseline tabular del Avance 3 (lo más probable), hay dos rutas: agrupación fenológica (reduce a ~10 clases) o pasar a modelos densos (Avance 4) y ensambles (Avance 5).

In [ ]:
if comparison_temporal is not None:
    best_row = comparison_temporal.row(0, named=True)
    best_f1 = float(best_row['f1_macro'])
    gap = F1_TARGET - best_f1
    status = '✓ cumplido' if gap <= 0 else f'✗ falta `{gap:+.4f}`'
    display(Markdown(
        f'**Estado actual del Avance 3**: el mejor modelo es '
        f'`{best_row["model"]}` con F1-macro = `{best_f1:.4f}`. '
        f'Target = `{F1_TARGET}` → {status}.'
    ))
    if gap > 0:
        display(Markdown(
            '**Plan para cerrar el gap**:\n\n'
            '1. Agrupación fenológica de clases minoritarias en `other_minor`.\n'
            '2. Modelos densos del Avance 4 (U-Net, U-TAE, TSViT, Swin-UNETR).\n'
            '3. Ensambles del Avance 5 (voting, bagging, stacking, blending) + Gemma 4 LoRA.'
        ))


## 7. Comparativa AlphaEarth vs Sentinel-2 crudo vs vector combinado

La sección 11 de `04_baseline.ipynb` entrena 3 modelos × 3 escenarios = **9 combinaciones** sobre exactamente el mismo conjunto de parcelas (inner join). El delta entre AlphaEarth y Sentinel-2 crudo cuantifica el valor incremental del embedding fundacional.

In [ ]:
scenarios_path = Path(COMPARISON_SCENARIOS_PATH)
if scenarios_path.exists():
    scenarios = pl.read_csv(scenarios_path)
    display(Markdown('**Comparativa de escenarios (9 filas)**:'))
    display(scenarios)
fig_path = env.repo / 'paper/figures/us-023-preview/04_baseline/comparison_barplot.png'
if fig_path.exists():
    display(Image(filename=str(fig_path)))
else:
    display(Markdown('> Figura comparativa no disponible.'))


## 8. Comparativa de extractores visuales: FarSLIP vs RemoteCLIP

`04_farslip_eval_pastis.ipynb` evalúa los dos extractores visuales con dos criterios independientes:

- Separabilidad lineal (LogReg + 5-fold estratificado).
- Separabilidad visual (UMAP 2D coloreado por clase).

In [ ]:
umap_figures = [
    ('UMAP FarSLIP por clase',
     'paper/figures/us-023-preview/04_farslip_eval_pastis/umap_farslip_by_class.png'),
    ('UMAP RemoteCLIP por clase',
     'paper/figures/us-023-preview/04_farslip_eval_pastis/umap_remoteclip_by_class.png'),
]
for label, rel_path in umap_figures:
    full_path = env.repo / rel_path
    if full_path.exists():
        display(Markdown(f'**{label}** — `{rel_path}`:'))
        display(Image(filename=str(full_path)))
    else:
        display(Markdown(f'> `{label}` no disponible.'))


## 9. Clustering fenológico sin coordenadas

`05_reencuadre_fenologico.ipynb` §6 demuestra que la firma fenológica pura organiza el dataset en arquetipos estacionales sin necesidad de coordenadas. Las curvas NDVI medias por cluster son interpretables agronómicamente.

In [ ]:
cluster_figures = [
    ('UMAP firma fenológica + KMeans',
     'paper/figures/us-023-preview/05_reencuadre/umap_clusters.png'),
    ('Curvas NDVI medias por cluster',
     'paper/figures/us-023-preview/05_reencuadre/cluster_ndvi_curves.png'),
]
for label, rel_path in cluster_figures:
    full_path = env.repo / rel_path
    if full_path.exists():
        display(Markdown(f'**{label}** — `{rel_path}`:'))
        display(Image(filename=str(full_path)))
    else:
        display(Markdown(f'> `{label}` no disponible.'))


## 10. Cuatro hipótesis del Avance 3 con decisiones consolidadas

Cada hipótesis se cierra con una decisión cuantitativa apoyada en la ablación. Las cifras son leídas del `ablation_table.parquet` que produjo `05_reencuadre`.

In [ ]:
import math

def _delta_or_none(name: str) -> float | None:
    if ablation_optional is None:
        return None
    rows = ablation_optional.filter(pl.col('feature_set') == name)
    if rows.height == 0:
        return None
    val = rows.get_column('delta_vs_full').to_list()[0]
    return float(val) if val is not None and not (isinstance(val, float) and math.isnan(val)) else None

def _f1_or_none(name: str) -> float | None:
    if ablation_optional is None:
        return None
    rows = ablation_optional.filter(pl.col('feature_set') == name)
    if rows.height == 0:
        return None
    val = rows.get_column('f1_macro').to_list()[0]
    return float(val) if val is not None and not (isinstance(val, float) and math.isnan(val)) else None

def _decision(delta: float | None, threshold: float = PROMOTE_THRESHOLD) -> str:
    if delta is None:
        return 'pendiente (sin datos)'
    if delta >= threshold:
        return f'promover (delta = {delta:+.4f} ≥ +{threshold:.3f})'
    if delta <= -threshold:
        return f'descartar (delta = {delta:+.4f} ≤ -{threshold:.3f})'
    return f'diferir a stacking (delta = {delta:+.4f} en zona neutra)'

geom_only_f1 = _f1_or_none('geom_only')
h1_decision = (
    f'descartar — `geom_only` F1-macro = `{geom_only_f1:.4f}` < 0.10' if geom_only_f1 is not None
    else 'descartar (decisión cualitativa, geom_only no evaluado)'
)

hypotheses = pl.DataFrame({
    'hipotesis': ['H-1 geom leakage', 'H-2 FarSLIP', 'H-3 pheno_text Gemini', 'H-4 firma espectral REP'],
    'descripcion': [
        'columnas geom_* son proxy de region (leakage espacial)',
        'FarSLIP (US-017) aporta señal complementaria al embedding tabular',
        'descripción fenológica textual con LLM aporta señal semántica',
        'Red Edge Position (Frampton 2013) captura forma espectral por época',
    ],
    'decision': [
        h1_decision,
        _decision(_delta_or_none('with_farslip')),
        _decision(_delta_or_none('with_pheno_text')),
        _decision(_delta_or_none('with_spectral_signature')),
    ],
})
display(Markdown('**Tabla H-1..H-4 — decisiones consolidadas**:'))
display(hypotheses)
hypotheses.write_parquet(env.reports_dir / 'decision_table.parquet')


## 11. Conjunto ganador con `select_winning_features`

Esta es la **única llamada a `select_winning_features` en todo el proyecto**. Recibe la `ablation_table` de `05_reencuadre`, promueve bloques con `delta >= +0.005`, descarta `geom_*` siempre, y persiste:

- `data/features/features_fused_winning_italy.parquet` — dataset filtrado a las columnas ganadoras.
- `data/features/features_fused_winning_italy.manifest.json` — lista nominal exacta de columnas (para que los modelos densos del Avance 4 y los ensambles del Avance 5 lean exactamente las mismas).

In [ ]:
from ml.features.winning_features import (
    select_winning_features,
    persist_winning_features,
)

if ablation_optional is None:
    raise FileNotFoundError(
        f'No se puede llamar a select_winning_features sin la tabla `{ABLATION_OPTIONAL_PATH_05}`. '
        'Ejecuta `05_reencuadre_fenologico.ipynb` antes de Avance3.'
    )

fused_path = Path(FUSED_PATH)
if not fused_path.exists():
    raise FileNotFoundError(
        f'No existe `{fused_path}`. Ejecuta `05_reencuadre_fenologico.ipynb` '
        '(genera el fused completo durante la materialización).'
    )
fused = pl.read_parquet(fused_path)

winning = select_winning_features(
    ablation_optional,
    available_cols=fused.columns,
    promote_threshold=PROMOTE_THRESHOLD,
    discard_geom=True,
)
display(Markdown('**Decisiones por bloque** (única fuente de verdad):'))
display(pl.DataFrame({
    'bloque': list(winning.decisions.keys()),
    'promovido': list(winning.decisions.values()),
}))
display(Markdown(
    f'**Conjunto ganador**: `{winning.name}` con `{len(winning.feature_cols)}` columnas.'
))
display(Markdown('### Rationale'))
display(Markdown(winning.rationale))

winning_path = persist_winning_features(
    winning, fused,
    output_path=WINNING_OUTPUT, overwrite=True,
)
display(Markdown(
    f'**Conjunto ganador guardado**: `{winning_path.relative_to(env.repo)}` · '
    f'**Manifest**: `{winning_path.with_suffix(".manifest.json").relative_to(env.repo)}`'
))

import json
manifest = json.loads(
    Path(WINNING_OUTPUT).with_suffix('.manifest.json').read_text(encoding='utf-8')
)
display(Markdown(f'**Número de características**: `{manifest["n_features"]}`'))
display(Markdown('**Características ganadoras** (primeras 40):'))
display(pl.Series('feature', manifest['feature_cols'][:40]).to_frame())


## 12. Trazabilidad MLflow consolidada

Cada notebook abre runs MLflow propios con tags `code_version` (SHA git) y `data_version` (hash DVC). Aquí listamos los experimentos para que cualquier auditor reabra y reproduzca.

In [ ]:
from ml.utils.mlflow_utils import resolve_tracking_uri, server_is_reachable
import mlflow
from mlflow.tracking import MlflowClient

# Misma logica defensiva que las celdas MLflow setup: si el server
# no responde, cae a `file:./mlruns` y consulta los runs locales.
_candidate_uri = resolve_tracking_uri(None, probe_server=False)
if _candidate_uri.startswith(('http://', 'https://')) and not server_is_reachable(_candidate_uri):
    mlflow_uri = 'file:./mlruns'
    display(Markdown(
        f'> Servidor MLflow `{_candidate_uri}` no responde. '
        'Consulto runs locales en `file:./mlruns`.'
    ))
else:
    mlflow_uri = _candidate_uri
mlflow.set_tracking_uri(mlflow_uri)
client = MlflowClient(tracking_uri=mlflow_uri)

experiments_av3 = [
    'baseline-04-tabular',
    'baseline-04b-pilot',
    'baseline-04c-ablation',
    'baseline-04-farslip-vs-remoteclip',
    'baseline-05-reencuadre',
]
summary_rows = []
for exp_name in experiments_av3:
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        summary_rows.append({
            'experimento': exp_name, 'n_runs': 0,
            'experiment_id': 'n/a', 'tracking_uri': mlflow_uri,
        })
        continue
    runs = client.search_runs(experiment_ids=[exp.experiment_id], max_results=200)
    summary_rows.append({
        'experimento': exp_name,
        'n_runs': len(runs),
        'experiment_id': exp.experiment_id,
        'tracking_uri': mlflow_uri,
    })
mlflow_summary = pl.DataFrame(summary_rows)
display(Markdown('**Resumen de experimentos MLflow del Avance 3**:'))
display(mlflow_summary)
mlflow_summary.write_parquet(env.reports_dir / 'mlflow_summary.parquet')


## 13. Referencias

- **Brown et al. (2025)** — *AlphaEarth Foundations: a Global Foundation Model for Earth*. Embedding satelital 64-dim.
- **Pelletier, Webb & Petitjean (2019)** — *TempCNN: Temporal Convolutional Neural Network for Satellite Image Time Series Classification*. DOI 10.3390/rs11050523.
- **Fawaz et al. (2020)** — *InceptionTime: Finding AlexNet for Time Series Classification*. DOI 10.1007/s10618-020-00710-y.
- **Garnot et al. (2021)** — *Panoptic Segmentation of Satellite Image Time Series with Convolutional Temporal Attention Networks (U-TAE)*. ICCV 2021.
- **Frampton et al. (2013)** — *Evaluating the capabilities of Sentinel-2 for quantitative estimation of biophysical variables in vegetation*. DOI 10.1016/j.isprsjprs.2013.04.007.
- **Wen et al. (2025)** — *Phenology Description is All You Need!*. Descripción fenológica textual + LLM.
- **Li et al. (2025)** — *FarSLIP: Patch-Level Distillation of CLIP for Remote Sensing*. arXiv:2511.14901.
- **Tang et al. (2024)** — *FarSLIP: Vineyard-aware CLIP Distillation*.
- **Chen et al. (2024)** — *RemoteCLIP: A Vision-Language Foundation Model for Remote Sensing*.
- **Lundberg & Lee (2017)** — *A Unified Approach to Interpreting Model Predictions (SHAP)*. NeurIPS.

**Atribuciones de licencia**: ver [`docs/licenses/DATA_LICENSE.md`](../../docs/licenses/DATA_LICENSE.md).

---

## Cierre

Con este cuaderno cerramos el Avance 3:

- **5 modelos reentrenados** sobre el conjunto ganador (3 tabulares + 2 temporales), con trazabilidad MLflow completa.
- **Las 5 preguntas oficiales** del Avance 3 respondidas con cifras concretas y figuras de los cuadernos previos.
- **Conjunto de características ganador** nombrado y persistido en `features_fused_winning_italy.parquet` + manifest JSON.
- **Cuatro hipótesis H-1..H-4** cerradas con decisión cuantitativa (promover, diferir, descartar).

**Lo que sigue (Avances 4 y 5)**:

- `notebooks/avance4_modelos.ipynb` consumirá el mismo `features_fused_winning_italy.parquet` y entrenará las 6 arquitecturas densas obligatorias: U-Net, DeepLabv3+, SegFormer-B2, U-TAE, TSViT (Paper 1), Swin-UNETR.
- `notebooks/avance5_ensambles.ipynb` construirá los 4 ensambles (voting top-3, bagging XGB+AlphaEarth, stacking + Gemma 4 26B-MoE LoRA, blending Optuna) sobre el mismo conjunto ganador.